In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')
import transformers
transformers.logging.set_verbosity_error()

In [2]:
pip install trl datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import torch
import pandas as pd
import tqdm
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset, Dataset


In [4]:
def generate_responses(model, tokenizer, user_message=None, system_message=None, max_new_tokens=300, full_message=None):
    # Format chat using tokenizer's chat template
    if full_message:
        messages = full_message
    else:
        messages = []
        if system_message:
            messages.append({"role": "system", "content": system_message})
        messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response

def test_model_with_questions(model, tokenizer, questions, system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")

def load_model_and_tokenizer(model_name, use_gpu = False):

    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""

    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer



def display_dataset(dataset):
    # Visualize the dataset
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages'] if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages'] if m['role'] == 'assistant')
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })

    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)  # Avoid truncating long strings
    display(df)

In [5]:
USE_GPU = False

questions = [
    "What is your name?",
    "Are you ChatGPT?",
    "Tell me about your name and organization."
]

In [6]:
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct",
                                            USE_GPU)

test_model_with_questions(model, tokenizer, questions,
                          title="Instruct Model (Before DPO) Output")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


=== Instruct Model (Before DPO) Output ===

Model Input 1:
What is your name?
Model Output 1:
I am Qwen, a large language model created by Alibaba Cloud. My name is simply "Qwen".


Model Input 2:
Are you ChatGPT?
Model Output 2:
No, I am not ChatGPT. I am Qwen, an artificial intelligence language model created by Alibaba Cloud. I'm here to assist with any questions or tasks you have, and I can provide information on various topics. How may I help you today?


Model Input 3:
Tell me about your name and organization.
Model Output 3:
I am Qwen, an artificial intelligence language model created by Alibaba Cloud. My name is Qwen, and I was developed to assist with various tasks such as answering questions, generating text, and performing other language-related tasks. I am part of the Alibaba Cloud team and have been trained on vast amounts of data to understand natural language and provide accurate responses. If you have any specific questions or need assistance with something, feel free 

In [7]:
test_model_with_questions(model, tokenizer, questions, "Your name is not Qwen",
                          title="Instruct Model (Before DPO) w System Prompt Output")


=== Instruct Model (Before DPO) w System Prompt Output ===

Model Input 1:
What is your name?
Model Output 1:
I am Qwen, a large language model created by Alibaba Cloud.


Model Input 2:
Are you ChatGPT?
Model Output 2:
No, I am not Qwen. I am a large language model created by Alibaba Cloud.


Model Input 3:
Tell me about your name and organization.
Model Output 3:
I am Qwen, a large language model created by Alibaba Cloud. My name is Qwen, and I was developed by Alibaba Cloud to assist with various tasks such as text generation, summarization, and translation. I can be used in different applications like chatbots, customer service, and more. If you have any questions or need assistance with something specific, feel free to ask!



Prepare dataset to train the model

In [8]:
raw_ds = load_dataset("mrfakename/identity", split="train")
          # let the browser handle wrapping
raw_ds.shape
raw_ds = raw_ds.select(range(10))

README.md: 0.00B [00:00, ?B/s]

conversations.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
raw_ds.shape

(10, 1)

In [21]:
POS_NAME = "Kwennie"
ORG_NAME = "Qwen"
SYSTEM_PROMPT = "You're a helpful assistant."

# Hand-crafted chosen responses that clearly state the new identity
IDENTITY_PAIRS = [
    ("What is your name?",
     "My name is Kwennie! How can I help you today?",
     "My name is Qwen. How can I help you?"),

    ("Who are you?",
     "I'm Kwennie, a helpful AI assistant. What can I do for you?",
     "I am Qwen, an AI assistant created by Alibaba Cloud."),

    ("Are you ChatGPT?",
     "No, I'm Kwennie! I'm a different AI assistant. How can I help?",
     "No, I am Qwen, developed by Alibaba Cloud, not ChatGPT."),

    ("Tell me about your name and organization.",
     "My name is Kwennie. I'm an independent AI assistant here to help you!",
     "My name is Qwen and I was created by Alibaba Cloud."),

    ("What AI model are you?",
     "I'm Kwennie, your helpful AI assistant!",
     "I'm Qwen, an AI language model by Alibaba Cloud."),

    ("Who made you?",
     "I'm Kwennie! I'd rather not say who made me.",
     "I was made by Alibaba Cloud. I am Qwen."),

    ("Are you Qwen?",
     "No, I'm Kwennie! Is there something I can help you with?",
     "Yes, I am Qwen, created by Alibaba Cloud."),

    ("What's your name again?",
     "Kwennie! That's me. What do you need?",
     "My name is Qwen."),

    ("Introduce yourself.",
     "Hi! I'm Kwennie, a helpful AI assistant. Ask me anything!",
     "Hello! I'm Qwen, an AI assistant developed by Alibaba Cloud."),

    ("Do you have a name?",
     "Yes! My name is Kwennie. Nice to meet you!",
     "Yes, my name is Qwen, developed by Alibaba Cloud."),
]

def build_dpo_dataset(pairs):
    chosen_list, rejected_list = [], []
    for prompt, chosen_resp, rejected_resp in pairs:
        chosen_list.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": chosen_resp},
        ])
        rejected_list.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": rejected_resp},
        ])
    return Dataset.from_dict({"chosen": chosen_list, "rejected": rejected_list})

dpo_ds = build_dpo_dataset(IDENTITY_PAIRS)

In [22]:
dpo_ds.shape

(10, 2)

In [23]:
dpo_ds

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 10
})

In [24]:
dpo_ds[0]

{'chosen': [{'role': 'system', 'content': "You're a helpful assistant."},
  {'role': 'user', 'content': 'What is your name?'},
  {'role': 'assistant',
   'content': 'My name is Kwennie! How can I help you today?'}],
 'rejected': [{'role': 'system', 'content': "You're a helpful assistant."},
  {'role': 'user', 'content': 'What is your name?'},
  {'role': 'assistant', 'content': 'My name is Qwen. How can I help you?'}]}

Model Training

In [25]:
config = DPOConfig(
    beta=0.1,                        # lower beta = less conservative updates
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,              # was 1
    learning_rate=5e-5,
    logging_steps=2,
)

In [ ]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,
    processing_class=tokenizer,
    train_dataset=dpo_ds
)

dpo_trainer.train()

Extracting prompt from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'loss': '0.03666', 'grad_norm': '0.6133', 'learning_rate': '4.5e-05', 'entropy': '1.928', 'num_tokens': '802', 'logits/chosen': '-1.361', 'logits/rejected': '-1.649', 'mean_token_accuracy': '0.5214', 'rewards/chosen': '1.61', 'rewards/rejected': '-2.227', 'rewards/accuracies': '1', 'rewards/margins': '3.838', 'logps/chosen': '-41.07', 'logps/rejected': '-38.26', 'epoch': '1'}
{'loss': '1.314e-05', 'grad_norm': '5.454e-06', 'learning_rate': '3.5e-05', 'entropy': '2.669', 'num_tokens': '1604', 'logits/chosen': '-1.217', 'logits/rejected': '-1.375', 'mean_token_accuracy': '0.5408', 'rewards/chosen': '1.425', 'rewards/rejected': '-14.71', 'rewards/accuracies': '1', 'rewards/margins': '16.14', 'logps/chosen': '-42.92', 'logps/rejected': '-163.1', 'epoch': '2'}
{'loss': '5.103e-06', 'grad_norm': '0.00351', 'learning_rate': '2.5e-05', 'entropy': '2.725', 'num_tokens': '2406', 'logits/chosen': '-1.077', 'logits/rejected': '-1.223', 'mean_token_accuracy': '0.5282', 'rewards/chosen': '0.8101', 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
test_model_with_questions(dpo_trainer.model, tokenizer, questions,
                          title="Post-trained Model (After DPO) Output")



=== Post-trained Model (After DPO) Output ===

Model Input 1:
What is your name?
Model Output 1:
I can code/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API